## SpectraGuru Raman Peak Assignment Table Extraction 🚧

This is an in-progess data extraction pipeline for extracting peak assignment data from Raman peak assignment tables in spectroscopy literature. It uses LLM-driven methods to extract and normalize Raman peak assignment data from a markdown file. Given a markdown file, it systematically extracts and normalizes:
1. Valid peak assignment table objects
2. A log of references
3. All raman-shift frequencies ordered from top-to-bottom (as displayed) in each peak assignment table
4. Mode assignments corresponding to each of the Raman-shift frequencies in each peak assignment table

The current version of this pipeline is useful for extracting peak assignment data from a narrow set of Raman literature inputs.

## LLM Instruction Methodology
To limit LLM output uncertainty, we introduce a variety of prompting and tool-calling techniques. These techniques have been shown to improve output consistency by ~70% across a range of models. For each prompt in this pipeline we adopt the following prompting structure:
#### Techniques
* <b>Role</b> - Assign the role of an expert spectroscopy practitioner.
* <b>Reasoning Steps</b> - Provide the guided path to reason to an effective output.
* (Some cases) <b>2 Pass Approach</b> - After conducting an initial extraction, have the model check its work against the raw data.
* <b>Givens</b> - State the inputs, definitions, additional notes, or any details that model should be aware of.
* <b>Task</b> - State the task for the model.
* <b>Rules and Guardrailes</b> - Give explicit directions for behaviours or assumptions that the model must avoid.
* <b>Output format</b> - The exact fields and types in structured format for the model output.
* <b>INPUT/OUTPUT Example</b> - One or two example cases of a correctly formated output for a given input.

#### Tools
* <b>Structured Output</B> - In the model config, we define our desired output schema with a BaseModel object.


## Results
We define our set of acceptable inputs to be markdown files that contain at least one <b>valid, Raman-only, peak assignment table</b> (Defined below). Constraining our initial set of inputs allows the current version of the pipeline to consistently provide correct outputs, preserving the fidelity of the inputs with 100% accuracy (Evaluated across 5 papers so far).


## Defining a <b> valid, Raman-only, peak assignment table</b>
An acceptable table must satisfy:
* Contains exactly one pair of frequency-assignment columns
    * Frequency column: A column of data illustrating the numerical frequencies for which peaks are observed.
    * Assignment column: A column of data illustrating the descriptive or conventional mode assignments corresponding to each frequency in the Frequency column.
* Is well-structured and unambiguous
    * No nested headers
    * No multi-column headers
    * Tabulated data
    * Assignments are interpretable without additional context that is not provided in the table
* Only contains Raman peak assignment data
    * No IR assignments
    * No SERS assignments
    * No additional assignments that are not strictly Raman

This definition is given to our LLM in the first step of the extraction process. 


## API Key and Input Config

In [ ]:
GEMINI_API_KEY="GEMINI_API_KEY"
input_file_path = "paper.md" # The path to the markdown file input
output_file_path = "deposit.json" # This should be a path to an empty json file

## Table Extraction - Content Validation
Extract valid Raman-only peak assignment tables. If no valid tables are found, we raise an exception.

In [2]:
from pydantic import BaseModel, Field
from typing import List
import json
from google import genai

class ValidRamanTable(BaseModel):
    table_title: str = Field(description="The exact title of the table the peak data is displayed in.")
    table_description: str = Field(description="The exact description of the table the peak data is displayed in.")
    class ConfigDict:
        extra = "forbid"
class ValidTables(BaseModel):
    number_valid_tables: int = Field(description="The number of valid Raman-only peak assignment tables found in the document.")
    tables: List[ValidRamanTable    ] = Field(description="A list of ValidRamanTable objects representing the Raman-only peak assignment tables found in the document.")
    class ConfigDict:
        extra = "forbid"

try: 

    prompt = """
        ### ROLE
        You are an expert Spectroscopy practitioner with 15 years of experience 
        performing research and publishing peer-reviewed studies in Raman and IR 
        spectroscopy. You have deep familiarity with vibrational mode assignment 
        conventions (Mulliken, Herzberg, Wilson Numbering, Wilson-Decius-Cross) and 
        can reliably identify well-structured Raman-only peak assignment tables.

        ---

        ### REASONING STEPS
        Before producing output, work through the following steps internally:

        1. Scan the entire document and identify every table present, noting its 
        title, description, headers, and column structure.
        2. For each table, check whether it contains a single frequency column with 
        numerical values (in cm⁻¹, although this is usually implied, use 
        your best judgement as an expert practitioner if units are not provided) and 
        a corresponding mode assignment column.
        3. Assess whether the table is self-contained — headers are unambiguous, 
        data is tabulated (not illustrated), and assignments can be read without 
        external context.
        4. Confirm the table covers Raman spectroscopy only — discard any table 
        that mixes Raman with IR, SERS, or any other technique.
        5. Apply strict mode: if a table only partially satisfies the criteria, 
        discard it silently.
        6. Count only the tables that pass all conditions as valid.
        7. For each valid table, record its exact title, exact description, and a 
        one-sentence explanation of why it qualifies.

        ---

        ### GIVENS
        - The input is a scientific paper provided in markdown format.
        - Tables may appear anywhere in the document — body, appendix, 
        supplementary sections, etc.
        - A table title is the short label preceding the description 
        (e.g., "Table 6.").
        - A table description is the explanatory text that follows the title 
        on the same line (e.g., "Raman spectrum of $C_2F_5I$").
        - Peak frequencies are reported as numerical values with units of cm⁻¹. 
        Column headers may use terminology such as "frequency", "wavenumber", 
        or "Raman shift."
        - Mode assignments may be descriptive (e.g., "C-H stretching", "ring 
        breathing") or use formal notation conventions (Mulliken, Herzberg, 
        Wilson Numbering, Wilson-Decius-Cross).

        ---

        ### TASK
        Identify all valid Raman-only peak assignment tables in the provided 
        markdown document and return a structured output containing the total 
        count and a summary object for each valid table.

        ---

        ### RULES / GUARDRAILS

        A valid Raman-only peak assignment table must satisfy ALL of the following:

        [I-1] Contains exactly one frequency-assignment pair:
                - A frequency column: numerical cm⁻¹ values with a header using 
                    "frequency", "wavenumber", or "Raman shift" (or equivalent).
                    Some tables may specify the header as a phase of matter, e.g. "gas",
                    "liquid", "solid", etc. This is acceptable. Sometimes the unit is 
                    not explicitly stated, in which case, use your best judgement as an 
                    expert practitioner.
                - A mode assignment column: a corresponding vibrational mode 
                    label for each frequency entry.
        [I-2] Is self-contained and unambiguous:
                - Headers clearly map each frequency to its assignment.
                - Data is tabulated — not represented as illustrations, figures, 
                    or embedded images.
                - Assignments are interpretable without external context 
                    (no footnotes, in-text references, or cross-table lookups 
                    required).
        [I-3] Covers Raman spectroscopy only — the table must not include 
                columns or data from IR, SERS, or any other spectroscopic 
                technique.

        If a table does not satisfy all three criteria, discard it silently. 
        Do not report or explain excluded tables in the output.

        ---

        ### OUTPUT FORMAT

        Line 1 — Final count:
        Number of valid Raman-only peak assignment tables: [N]

        Then output one object per valid table in the following structure:

        - Table Title:       [Exact title text as it appears in the document]
        - Table Description: [Exact description text as it appears in the document]
        - Validity:          [One sentence explaining why this table satisfies 
                            I-1, I-2, and I-3]

        ---

        ### EXAMPLE

        INPUT (excerpt):
        Table 3. Raman band assignments for naphthalene

        | Raman Shift (cm⁻¹) | Assignment         |
        |--------------------|--------------------|
        | 763                | C-H out-of-plane   |
        | 1004               | Ring breathing     |
        | 1382               | C-C stretching     |

        OUTPUT:
        Number of valid Raman-only peak assignment tables: 1

        - Table Title:       Table 3.
        - Table Description: Raman band assignments for naphthalene
        - Validity:          Contains a clearly labeled Raman shift column with 
                            cm⁻¹ values paired with unambiguous mode assignments, 
                            is fully self-contained, and covers Raman spectroscopy 
                            only.

        ---

        Now examine the following markdown file and apply the criteria above:

    """

    

    print("\n Collecting valid Raman-only peak assignment tables...")

    client = genai.Client(api_key=GEMINI_API_KEY)
    my_file = client.files.upload(file=input_file_path)
    response = client.models.generate_content(
        model="gemini-3-flash-preview", 
        contents=[
            prompt,
            my_file
        ],
        config={
            "response_mime_type": "application/json",
            "response_json_schema": ValidTables.model_json_schema()
        }
        )

    raman_tables = ValidTables.model_validate_json(response.text)
    
    print(json.dumps(raman_tables.model_dump(), indent=2))
    obj = json.dumps(raman_tables.model_dump(), indent=2)
    response_obj = json.loads(obj)
    # print(f"\nFound {response_obj['number_valid_tables']} valid peak assignment tables in the document.")
except Exception as e:
    print(f"Error: {e}")

# Deposit Valid Tables in our deposit.json file
from pathlib import Path

path = Path(output_file_path)
new_data = raman_tables.model_dump() 
if path.exists() and path.read_text(encoding="utf-8").strip():
    with open(path, "r", encoding="utf-8") as f:
        existing = json.load(f)
else:
    existing = {"raman_peak_assignment_tables": []}
    
if new_data["number_valid_tables"] == 0:
    raise Exception("No valid Raman-only peak assignment tables found in the document.")

existing.setdefault("number_valid_raman_only_tables", new_data.get("number_valid_tables", 0))
existing.setdefault("raman_peak_assignment_tables", []).extend(new_data.get("tables", []))

with open(path, "w", encoding="utf-8") as f:
    json.dump(existing, f, indent=2)
    f.write("\n")


{
  "number_valid_tables": 3,
  "tables": [
    {
      "table_title": "Table I .",
      "table_description": "The Raman Spectrum of  $\\text{CF}_3\\text{CH}_2\\text{F}$  Liquid (From densitometer tracing of 45 hours exposure)"
    },
    {
      "table_title": "Table II.",
      "table_description": "The Raman Spectrum of  $\\text{CF}_3\\text{CH}_2\\text{Br}$"
    },
    {
      "table_title": "Table III.",
      "table_description": "The Raman Spectrum of  $CF_3CH_2F$"
    }
  ]
}


## Log all unique references in the document

In [4]:
class Reference(BaseModel):
    reference_num: int = Field(description="The number of the reference displayed in the file. Usually between an open and closing pair of square brackets.")
    raw_reference: str = Field(description="The entire reference as it appears in the line in the file.")
    class ConfigDict:
        extra = "forbid"

class ReferenceLog(BaseModel):
    references: List[Reference] = Field(description="A list of references extracted from the file.")
    class ConfigDict:
        extra = "forbid"

prompt = """
    ### ROLE
    You are a precise scientific literature parser. Your sole function is to identify 
    and extract every unique reference entry from a scientific paper provided in 
    markdown format. You have expert familiarity with academic citation formats 
    (APA, MLA, IEEE, Vancouver, Chicago, etc.) and can reliably detect reference 
    entries regardless of their formatting style.

    ---

    ### REASONING STEPS
    Before producing output, work through the following steps internally:

    1. Scan the entire document to locate the references section (commonly headed 
    "References", "Bibliography", "Works Cited", "Literature Cited", etc.)
    2. Identify every numbered reference entry — each is typically preceded by a 
    number in square brackets (e.g. [1], [2]) or another numbering convention.
    3. Check for any inline citations or references that may appear outside the 
    formal references section and determine if they correspond to a listed 
    reference or are standalone entries not yet captured.
    4. De-duplicate: if the same reference appears more than once, include it only 
    once using its first occurrence.
    5. Preserve each reference exactly as it appears in the source — do not 
    correct, reformat, or summarize.
    6. Map each entry to the required JSON schema fields before writing output.

    ---

    ### GIVENS
    - The input is a scientific paper in markdown format.
    - References are numbered and enclosed in square brackets, e.g. [1], [2], [23].
    - The raw reference text may contain markdown formatting such as _italics_, 
    **bold**, or `code spans` — preserve all of it exactly.
    - The reference number to use is the integer inside the square brackets at the 
    start of the entry.
    - You must extract ALL unique references present in the document — do not 
    skip, truncate, or summarize any.

    ---

    ### TASK
    Extract every unique reference from the provided scientific paper and return 
    them as a JSON array. Each element in the array must conform exactly to the 
    schema defined in the OUTPUT FORMAT section below.

    ---

    ### RULES / GUARDRAILS
    - DO NOT infer, hallucinate, or fabricate any references not present in the 
    source document.
    - DO NOT correct typos, fix formatting, or alter the raw reference text in 
    any way.
    - DO NOT include duplicate references — if a reference number appears more 
    than once with identical content, include it only once.
    - If two entries share the same reference number but have different content, 
    flag this as a conflict by appending a note field: "conflict": true.
    - DO NOT include any explanation, commentary, or prose outside of the JSON 
    output block.
    - If no references are found in the document, return an empty array: [].

    ---

    ### OUTPUT FORMAT
    Return a single JSON array. Each object in the array must conform to this 
    schema exactly:

    {
    "reference_num": <integer>,       // The integer from inside the [N] bracket
    "raw_reference": "<string>"       // The full reference line exactly as it 
                                        // appears in the source, including the 
                                        // leading [N] bracket and all markdown
    }

    No additional fields are permitted. The array should be ordered by 
    reference_num ascending.

    ---

    ### EXAMPLE

    INPUT (excerpt from markdown file):
    ...
    [1] J. R. NIELSEN, C. Y. LIANG, R. M. SMITH and D. C. SMITH, _J. Chem. Phys._ **21**, 383 (1953).
    [2] A. EINSTEIN, _Ann. Phys._ **17**, 891 (1905).
    ...

    OUTPUT:
    ```json
    [
    {
        "reference_num": 1,
        "raw_reference": "[1] J. R. NIELSEN, C. Y. LIANG, R. M. SMITH and D. C. SMITH, _J. Chem. Phys._ **21**, 383 (1953)."
    },
    {
        "reference_num": 2,
        "raw_reference": "[2] A. EINSTEIN, _Ann. Phys._ **17**, 891 (1905)."
    }
    ]
    ```

    ---

    Now extract all unique references from the following scientific paper:

"""

try :
    client = genai.Client(api_key=GEMINI_API_KEY)
    my_file = client.files.upload(file=input_file_path)
    response = client.models.generate_content(
        model="gemini-3-flash-preview", 
        contents=[
            prompt,
            my_file
        ],
        config={
            "response_mime_type": "application/json",
            "response_json_schema": ReferenceLog.model_json_schema()
        }
    )
    reference_log = ReferenceLog.model_validate_json(response.text)
    print(json.dumps(reference_log.model_dump(), indent=2))
    if reference_log.model_dump().get("references", []):
        print(f"{len(reference_log.model_dump().get('references', []))} references detected!")
    if not reference_log.model_dump().get("references", []):
        raise Exception("No references found in the document.")
except Exception as e:
    print(f"Error: {e}")

# Deposit References in our deposit.json file
path = Path(output_file_path)
new_data = reference_log.model_dump()  # dict with "references": [...]
if path.exists() and path.read_text(encoding="utf-8").strip():
    with open(path, "r", encoding="utf-8") as f:
        existing = json.load(f)
else:
    existing = {"references": []}

existing.setdefault("references", []).extend(new_data.get("references", []))

with open(path, "w", encoding="utf-8") as f:
    json.dump(existing, f, indent=2)
    f.write("\n")

{
  "references": [
    {
      "reference_num": 1,
      "raw_reference": "1. A. L. HENNE, R. M. ALM, AND M. SMOOK, *J. Am. Chem. Soc.* **70**, 1968 (1948)."
    },
    {
      "reference_num": 2,
      "raw_reference": "2. W. F. EDGELL AND L. PARTS, *J. Am. Chem. Soc.* **77**, 4899 (1955)."
    },
    {
      "reference_num": 3,
      "raw_reference": "3. C. E. MAY, Ph.D. thesis, Purdue University, 1953."
    },
    {
      "reference_num": 4,
      "raw_reference": "4. R. M. POTTER, Ph.D. thesis, Purdue University, 1953."
    },
    {
      "reference_num": 5,
      "raw_reference": "5. J. R. NIELSEN, H. H. CLAASEN, AND D. C. SMITH, *J. Chem. Phys.* **18**, 1471 (1950)."
    },
    {
      "reference_num": 6,
      "raw_reference": "6. J. R. NIELSEN, C. Y. LIANG, AND D. C. SMITH, *J. Chem. Phys.* **21**, 1060 (1953)."
    },
    {
      "reference_num": 7,
      "raw_reference": "7. E. H. BORNEMAN, Ph.D. thesis, Purdue University. 1952."
    },
    {
      "reference_num": 8,
      

## Extract Peak Rows as objects

In [5]:
from typing import Optional
from pydantic import ConfigDict, model_validator


class PeakRow(BaseModel):
    peak: Optional[int] = Field(description="The single integer value of the peak frequency in units of cm⁻¹ (if only one is given).")
    peak_high: Optional[int] = Field(description="The upper bound of the range of the peak frequency in units of cm⁻¹ (if a range is given).")
    peak_low: Optional[int] = Field(description="The lower bound of the range of the peak frequency in units of cm⁻¹ (if a range is given).")
    peak_width: Optional[int] = Field(description="The width of the peak measured at half the peak height, where intensity is half of the maximum intensity of the peak. (FWHM 'Full Width at Half Maximum')(usually in cm^-1)")
    relative_intensity: Optional[str] = Field(description="The relative intensity of the peak, usually given as a descriptive label like 'vvs','vs', 's', 'm', 'mw', 'w', 'vw', 'vvw', meaning (very very strong), (very strong), (strong), (medium), (medium weak), (weak), (very weak), (very very weak) respectively.")
    additional_notes: Optional[str] = Field(description="Additional details about the peak assignment from the literature. Include polarization details (p, dp, etc. meaning polarized, depolarized, etc. respectively) with keys to specify if necessary. Also include any baseline correction if stated in literature.")

    # Explicit schema validation to ensure correct formatting
    model_config = ConfigDict(
        extra="forbid",
        json_schema_extra={
            "anyOf": [
                {"required": ["peak"]},
                {"required": ["peak_low", "peak_high"]},
            ]
        },
    )

    # Model validation at runtime to ensure correct formatting
    @model_validator(mode="after")
    def _require_peak_or_range(self):
        if self.peak is not None:
            return self
        if self.peak_high is not None and self.peak_low is not None:
            return self
        raise ValueError("A PeakRow must contain either 'peak', or both 'peak_high' and 'peak_low'.")


class Rows(BaseModel):
    rows: List[PeakRow] = Field(
        description="A list of PeakRow objects representing the individual peak-assignment rows in the table."
    )

    class ConfigDict:
        extra = "forbid"

# Get the list of table titles and descriptions
path = Path(output_file_path)
new_data = reference_log.model_dump()  # dict with "references": [...]
if path.exists() and path.read_text(encoding="utf-8").strip():
    with open(path, "r", encoding="utf-8") as f:
        existing = json.load(f)
        tables = existing.get("raman_peak_assignment_tables", [])
else:
    raise Exception("The document could not be opened.")

if len(tables) == 0:
    raise Exception("No tables found in the document.")

# print(tables)

def construct_prompt_peak_rows(table_title, table_description):

    base_prompt = """

    ### ROLE
    You are an expert Spectroscopy practitioner with 15 years of experience 
    performing research and publishing peer-reviewed studies in Raman and IR 
    spectroscopy. You have deep familiarity with vibrational mode assignment 
    conventions (Mulliken, Herzberg, Wilson Numbering, Wilson-Decius-Cross) and 
    can precisely extract, normalize, and verify peak assignment row data from 
    structured spectroscopy tables.

    ---

    ### REASONING STEPS
    Before producing output, work through the following steps internally:

    PASS 1 — EXTRACTION
    1. Locate the target table in the document using the provided title 
        and description.
    2. Read each data row in the table from top to bottom. Skip header 
        rows and any rows that do not contain peak frequency data.
    3. For each row, parse and normalize the following fields:
        - peak: a single integer cm⁻¹ value (if only one frequency given).
        - peak_low / peak_high: the lower and upper integer bounds 
            (if a frequency range is given).
        - peak_width: the FWHM value in cm⁻¹ if stated.
        - relative_intensity: the exact description of the relative intensity of the peak. 
            If only given labels, include the label and a key to specify if necessary. (Key to 
            use is specified in the GIVENS section of the prompt.)
        - additional_notes: polarization details, baseline corrections, 
            or any other qualifying information attached to the row that is not related to the vibrational mode assignment.
    4. Produce a candidate list of PeakRow objects.

    PASS 2 — DEDUPLICATION AND VERIFICATION
    5. Compare the candidate list back against the original table row by row.
    6. Confirm that every data row in the table is represented exactly once 
        in the candidate list — no row is missing, duplicated, or fabricated.
    7. Confirm that all field values accurately reflect what appears in 
        the table — correct any extraction errors found.
    8. Produce the final verified list.

    ---

    ### GIVENS
    - You will receive the full markdown document and the exact title and 
    description of the target table.
    - The target table is uniquely identified by matching both its title 
    AND description simultaneously.
    - Peak frequencies are integers in cm⁻¹. A range is indicated by a 
    dash or similar separator (e.g. "1200-1250").
    - Relative intensity labels follow standard spectroscopy shorthand:
        vvs = very very strong  |  vs  = very strong  |  s   = strong
        m   = medium            |  mw  = medium weak   |  w   = weak
        vw  = very weak         |  vvw = very very weak
    - Polarization labels follow standard shorthand:
        p = polarized  |  dp = depolarized  |  ap = anomalously polarized
    - A PeakRow must contain EITHER peak OR both peak_low AND peak_high. 
    All other fields are optional.
    - Do not infer, hallucinate, or fill in any field that is not explicitly 
    stated in the table row.

    ---

    ### TASK
    Given the full markdown document and the title and description of a 
    specific table within it:

    1. Locate the target table.
    2. Extract every peak assignment data row and normalize each into a 
    PeakRow object.
    3. Deduplicate and verify the extracted rows across two passes.
    4. Return the final verified list as a Rows object containing a list 
    of PeakRow objects.

    ---

    ### RULES / GUARDRAILS
    - Extract data ONLY from the identified target table — ignore all other 
    tables in the document.
    - DO NOT infer, hallucinate, or populate any field not explicitly present 
    in the row.
    - DO NOT merge rows — each data row in the table maps to exactly one 
    PeakRow object.
    - If a frequency value is given as a range, populate peak_low and 
    peak_high and leave peak as null.
    - If a frequency value is a single integer, populate peak and leave 
    peak_low and peak_high as null.
    - Intensity labels, polarization details, and any other qualifying 
    text must be captured — intensity in relative_intensity, all other 
    qualifying details in additional_notes.
    - DO NOT include any explanation, commentary, or prose outside of the 
    JSON output block.

    ---

    ### OUTPUT FORMAT
    Return a single JSON object conforming to the Rows schema:

    {
    "rows": [
        {
        "peak": <integer | null>,
        "peak_high": <integer | null>,
        "peak_low": <integer | null>,
        "peak_width": <integer | null>,
        "relative_intensity": <string | null>,
        "additional_notes": <string | null>
        },
        ...
    ]
    }

    All six fields must be present in every object. Use null for any field 
    that is not present in the source row.

    ---

    ### EXAMPLE

    INPUT:

    Table Title:       Table 4.
    Table Description: Raman spectrum of liquid CHF₂Cl

    | Liquid         | Assignment           |
    | -------------- | -------------------- |
    | 1321 w, depol. | $\nu_{12}$ $a''$     |
    | 1234 w, pol?   | $\nu_1$ $a'$         |
    | 1210 m, pol.   | $\nu_2$ $a'$         |

    OUTPUT:

    ```json
    {
    "rows": [
        {
        "peak": 1321,
        "peak_high": null,
        "peak_low": null,
        "peak_width": null,
        "relative_intensity": "w",
        "additional_notes": "depol."
        },
        {
        "peak": 1234,
        "peak_high": null,
        "peak_low": null,
        "peak_width": null,
        "relative_intensity": "w",
        "additional_notes": "pol? (polarization uncertain)"
        },
        {
        "peak": 1210,
        "peak_high": null,
        "peak_low": null,
        "peak_width": null,
        "relative_intensity": "m",
        "additional_notes": "pol."
        }
    ]
    }
    ```

    ---

    Now extract the peak assignment rows from the following document and table:
    """

    complete_prompt = base_prompt + f"\n\nTable Title:{table_title}\nTable Description: {table_description}"
    return complete_prompt

print("Gathering peak assignment rows...")
try:
    client = genai.Client(api_key=GEMINI_API_KEY)
    my_file = client.files.upload(file=input_file_path)

    for i in range(len(tables)):
        prompt = construct_prompt_peak_rows(tables[i]["table_title"], tables[i]["table_description"])
        response = client.models.generate_content(
            model="gemini-3-flash-preview", 
            contents=[
                prompt,
                my_file
            ],
            config={
                "response_mime_type": "application/json",
                "response_json_schema": Rows.model_json_schema()
            }
        )
        rows = Rows.model_validate_json(response.text)
        # Append the new rows list to the existing table object
        tables[i]["rows"] = rows.model_dump()["rows"]
        # Write the updated tables list to the JSON file
        # existing["raman_peak_assignment_tables"] = tables
        with open(path, "w", encoding="utf-8") as f:
            json.dump(existing, f, indent=2)
            f.write("\n")

except Exception as e:
    print(f"Error: {e}")

Gathering peak assignment rows...


## Enrich peak rows with vibrational mode assignments

In [6]:
class VibrationalMode(BaseModel):
    notation_convention: Optional[str] = Field(description="The notation convention used to assign the mode index (Mulliken, Herzberg, Wilson Numbering, Wilson-Decius-Cross, or descriptive)")
    assignment_label: Optional[str] = Field(description="The shorthand identifier used to assign the vibrational mode in literature (v4, v10, etc.). This follows specific naming conventions (Mulliken, Herzberg, Wilson Numbering, Wilson-Decius-Cross, or descriptive)")
    mulliken_label: Optional[str] = Field(description="The irreducible label of the vibrational symmetry in machine-readable, plain text, following the Mulliken notation convention (A1g, B2u, E_prime, A1g_sym, etc.)")
    mulliken_symmetry_label: Optional[str] = Field(description="The symmetry label given to the vibrational mode using the Mulliken labeling convention. (A', a'', B, b, T, t, e, E, g-sub, u-sub)")
    descriptive_symmetry_label: Optional[str] = Field(description="The descriptive label of the vibrational symmetry (symmetric, antisymmetric, unsymmetrical, degenerate).")
    descriptive_symmetry_label_raw: Optional[str] = Field(description="The exact text used to describe the vibrational symmetry (sym., antisym., deg.).")
    descriptive_motion_label: Optional[str] = Field(description="The descriptive label of the vibrational motion (stretch, bend, twist, rock, scissoring, deformation, twisting-rocking-rocking-twisting, etc.)")
    geometric_motion_label: Optional[str] = Field(description="The descriptive label of the vibrational geometry (in-plane, out-of-plane, radial, tangential, etc.).")
    phase_label: Optional[str] = Field(description="The descriptive label of the vibrational phase (in-phase, out-of-phase, etc.).")
    moiety: Optional[str] = Field(description="The distinct, identifiable bond in the molecule that is analyzed in the table (C—C, N=C=N, etc.) (Use SMILES format).")
    vibrational_mode_description: Optional[str] = Field(description="The exact raw text that is used to report the vibrational mode, if applicable (sym. str., C—C skel. def., etc.) (raw text. non-standardized).")
    strucutural_position: Optional[str] = Field(description="Raw descriptive text to describe where a group sits within a molecule. (terminal, internal, axial, endocyclic, exocyclic, backbone) ")
    raw_assignment: str = Field(description="The exact raw text used to report the vibrational mode assignment in the table (raw text. non-standardized).")

    model_config = ConfigDict(
        extra = "forbid"
    )

class VibrationalModeList(BaseModel):
    vibrational_modes: List[VibrationalMode] = Field(description="A list of VibrationalMode objects representing the vibrational modes in the document.")

    class ConfigDict:
        extra = "forbid"

def construct_prompt_vibrational_modes(table_title, table_description, peak_rows):
    base_prompt = """
    ### ROLE
    You are an expert Spectroscopy practitioner with 15 years of experience 
    performing research and publishing peer-reviewed studies in Raman and IR 
    spectroscopy. You have deep familiarity with vibrational mode assignment 
    conventions (Mulliken, Herzberg, Wilson Numbering, Wilson-Decius-Cross) and 
    can precisely identify, decompose, and normalize vibrational mode assignments 
    from structured spectroscopy tables.
    ---
    ### REASONING STEPS
    Before producing output, work through the following steps internally:
    PASS 1 — EXTRACTION
    1. Locate the target table in the document using the provided title and 
       description. Match both simultaneously.
    2. Count the rows in the provided peak row list. This count defines the 
       exact number of VibrationalMode objects you must produce — no more, 
       no less.
    3. For each peak row in the provided list (in order, top to bottom), 
       identify its corresponding table row by matching the peak frequency 
       value to the frequency cell on that line of the table.
    4. Identify all columns in the table that contain vibrational mode 
       assignment data. The primary assignment column is typically labeled 
       "Assignment" or equivalent. Additional columns (e.g., "Symmetry", 
       "Description", "Motion") may also carry assignment-relevant data.
    5. For each matched table row, extract and normalize the following fields 
       from all relevant columns:
        - raw_assignment: The complete, unmodified text of the primary 
          assignment cell exactly as it appears in the table.
        - assignment_label: The shorthand mode index identifier (ν12, ν4, 
          v10, etc.) if present.
        - notation_convention: The naming convention used for the assignment 
          label (Wilson Numbering, Mulliken, Herzberg, Wilson-Decius-Cross, 
          or descriptive) if determinable.
        - mulliken_symmetry_label: The symmetry species shorthand in Mulliken 
          notation (a', a'', A1, B2, E, T, etc.) if explicitly present.
        - mulliken_label: The full machine-readable irreducible representation 
          label (A1g, B2u, E_prime, etc.) if explicitly present.
        - descriptive_symmetry_label: The normalized symmetry classification 
          (symmetric, antisymmetric, degenerate, unsymmetrical) — populate 
          if explicitly stated OR inferable from the Mulliken symmetry label 
          using the inference rules in GIVENS.
        - descriptive_symmetry_label_raw: The exact raw text used to describe 
          the symmetry (sym., antisym., deg., etc.) if explicitly present.
        - descriptive_motion_label: The type of vibrational motion 
          (stretch, bend, rock, deformation, torsion, scissoring, wag, 
          twisting, etc.) if explicitly present.
        - geometric_motion_label: The geometric qualifier 
          (in-plane, out-of-plane, radial, tangential, etc.) if present.
        - phase_label: The phase descriptor (in-phase, out-of-phase) if present.
        - moiety: The specific bond or chemical group involved, expressed in 
          bond notation using SMILES conventions (C—C, [N]=[C]=[N], etc.) 
          if explicitly present.
        - vibrational_mode_description: The raw text description of the 
          vibrational mode as it appears in the table 
          (e.g., "CF3 stretch", "sym. str.", "C—C skel. def.") if present.
        - strucutural_position: Raw descriptive text for structural position 
          (terminal, internal, axial, endocyclic, exocyclic, backbone, etc.) 
          if explicitly present; use empty string ("") if not stated.
    6. Produce a candidate list of VibrationalMode objects, ordered to match 
       the input peak rows list exactly (index 0 of peak rows → index 0 of 
       vibrational modes, etc.).
    PASS 2 — VERIFICATION
    7. Compare each VibrationalMode in the candidate list back against its 
       corresponding table row.
    8. Confirm that every peak row maps to exactly one VibrationalMode 
       object — no row is missing, duplicated, or fabricated.
    9. Confirm the total count of VibrationalMode objects equals the total 
       count of input peak rows.
    10. Verify all field values accurately reflect what is present in the 
        table row — correct any extraction or inference errors found.
    11. Produce the final verified list.
    ---
    ### GIVENS
    - You will receive the full markdown document, the exact title and 
      description of the target table, and an ordered JSON array of normalized 
      peak row objects produced by the previous extraction step.
    - The target table is uniquely identified by matching both its title AND 
      description simultaneously.
    - The vibrational mode assignment for each peak row is located on the 
      same line in the table as the peak frequency.
    - The primary assignment data is typically under a column labeled 
      "Assignment" or equivalent. Some tables include additional columns 
      (e.g., a separate "Symmetry", "Description", or "Motion" column) — 
      extract relevant fields from all such columns.
    - Notation convention identification:
        Wilson Numbering:       mode label is ν (nu) followed by an 
                                integer subscript (ν1, ν12, ν18, etc.).
        Herzberg:               mode label uses a different subscript 
                                or superscript convention than Wilson.
        Mulliken:               mode labeled directly with a symmetry 
                                species (A1, B2u, Eg, etc.) without ν.
        Descriptive:            assignment given as plain natural language 
                                (e.g., "C—H stretch", "ring breathing", 
                                "CF3 deformation").
    - Descriptive symmetry inference from Mulliken symmetry labels 
      (ONLY apply when explicit descriptive text is absent):
        a', A, A1, A1g, Ag   →  symmetric
        a'', B, B1, B2, Au   →  antisymmetric
        E, Eg, Eu, T, etc.   →  degenerate
    - moiety values should use bond notation following SMILES conventions. 
      Preserve the exact bond type (—, =, ≡) and atom labels as written 
      in the source.
    ---
    ### TASK
    Given the full markdown document, the title and description of a specific 
    table, and the ordered JSON list of normalized peak row objects:
    1. Locate the target table.
    2. For each peak row in the input list, find its corresponding table row 
       and extract all vibrational mode assignment information from it.
    3. Normalize each extracted assignment into a VibrationalMode object.
    4. Verify the complete list across two passes.
    5. Return the final verified list as a VibrationalModeList.
    ---
    ### RULES / GUARDRAILS
    - Extract assignment data ONLY from the identified target table — ignore 
      all other tables in the document.
    - The output list MUST contain exactly the same number of VibrationalMode 
      objects as there are peak rows in the input list.
    - If a table row has no assignment data, return a VibrationalMode with 
      raw_assignment set to an empty string ("") and all optional fields null.
    - DO NOT infer, hallucinate, or populate any field that is not explicitly 
      present in the table row, with the single exception of 
      descriptive_symmetry_label, which may be inferred from the Mulliken 
      symmetry label using the rules in GIVENS.
    - DO NOT merge rows — each peak row maps to exactly one VibrationalMode.
    - raw_assignment must always contain the complete, unmodified text of the 
      primary assignment cell exactly as it appears in the table 
      (preserve all LaTeX, markdown, subscripts, and special characters).
    - strucutural_position must always be populated — use an empty string 
      ("") if no structural position is stated in the row.
    - DO NOT include any explanation, commentary, or prose outside of the 
      JSON output block.
    ---
    ### OUTPUT FORMAT
    Return a single JSON object conforming to the VibrationalModeList schema:
    {
      "vibrational_modes": [
        {
          "notation_convention": <string | null>,
          "assignment_label": <string | null>,
          "mulliken_label": <string | null>,
          "mulliken_symmetry_label": <string | null>,
          "descriptive_symmetry_label": <string | null>,
          "descriptive_symmetry_label_raw": <string | null>,
          "descriptive_motion_label": <string | null>,
          "geometric_motion_label": <string | null>,
          "phase_label": <string | null>,
          "moiety": <string | null>,
          "vibrational_mode_description": <string | null>,
          "strucutural_position": <string>,
          "raw_assignment": <string>
        },
        ...
      ]
    }
    All 13 fields must be present in every object. Use null for any optional 
    field not present in the source row. strucutural_position and raw_assignment 
    must never be null.
    ---
    ### EXAMPLE
    INPUT:
    Table Title:       Table 4.
    Table Description: Raman spectrum of liquid CHF₂Cl
    Peak Rows (JSON):
    [
      {
        "peak": 1321, "peak_high": null, "peak_low": null,
        "peak_width": null, "relative_intensity": "w",
        "additional_notes": "depol."
      },
      {
        "peak": 1234, "peak_high": null, "peak_low": null,
        "peak_width": null, "relative_intensity": "w",
        "additional_notes": "pol?"
      },
      {
        "peak": 1210, "peak_high": null, "peak_low": null,
        "peak_width": null, "relative_intensity": "m",
        "additional_notes": "pol."
      }
    ]
    Target table (as it appears in the markdown document):
    Table 4. Raman spectrum of liquid CHF₂Cl
    | Liquid          | Assignment           | Description       |
    | --------------- | -------------------- | ----------------- |
    | 1321 w, depol.  | $\nu_{12}$ $a''$     | CF3 stretch       |
    | 1234 w, pol?    | $\nu_1$ $a'$         | CF3 stretch       |
    | 1210 m, pol.    | $\nu_2$ $a'$         | CF2 stretch, sym. |
    OUTPUT:
    ```json
    {
      "vibrational_modes": [
        {
          "notation_convention": "Wilson Numbering",
          "assignment_label": "ν12",
          "mulliken_label": null,
          "mulliken_symmetry_label": "a''",
          "descriptive_symmetry_label": "antisymmetric",
          "descriptive_symmetry_label_raw": null,
          "descriptive_motion_label": "stretch",
          "geometric_motion_label": null,
          "phase_label": null,
          "moiety": null,
          "vibrational_mode_description": "CF3 stretch",
          "strucutural_position": "",
          "raw_assignment": "$\\nu_{12}$ $a''$"
        },
        {
          "notation_convention": "Wilson Numbering",
          "assignment_label": "ν1",
          "mulliken_label": null,
          "mulliken_symmetry_label": "a'",
          "descriptive_symmetry_label": "symmetric",
          "descriptive_symmetry_label_raw": null,
          "descriptive_motion_label": "stretch",
          "geometric_motion_label": null,
          "phase_label": null,
          "moiety": null,
          "vibrational_mode_description": "CF3 stretch",
          "strucutural_position": "",
          "raw_assignment": "$\\nu_1$ $a'$"
        },
        {
          "notation_convention": "Wilson Numbering",
          "assignment_label": "ν2",
          "mulliken_label": null,
          "mulliken_symmetry_label": "a'",
          "descriptive_symmetry_label": "symmetric",
          "descriptive_symmetry_label_raw": "sym.",
          "descriptive_motion_label": "stretch",
          "geometric_motion_label": null,
          "phase_label": null,
          "moiety": null,
          "vibrational_mode_description": "CF2 stretch, sym.",
          "strucutural_position": "",
          "raw_assignment": "$\\nu_2$ $a'$"
        }
      ]
    }
    ```
    ---
    Now extract the vibrational mode assignments from the following document and table:
    """
    peak_rows_json = json.dumps(peak_rows, indent=2)
    complete_prompt = (
        base_prompt
        + f"\n\nTable Title: {table_title}"
        + f"\nTable Description: {table_description}"
        + f"\n\nPeak Rows (JSON):\n{peak_rows_json}"
    )
    return complete_prompt

with open('deposit.json', "r", encoding="utf-8") as f:
    existing = json.load(f)
    tables = existing.get("raman_peak_assignment_tables", [])
print(tables)


try:
    client = genai.Client(api_key=GEMINI_API_KEY)
    my_file = client.files.upload(file=input_file_path)


    for i in range(len(tables)):
        prompt = construct_prompt_vibrational_modes(tables[i]["table_title"], tables[i]["table_description"], tables[i]["rows"])
        response = client.models.generate_content(
            model="gemini-3-flash-preview", 
            contents=[
                prompt,
                my_file
            ],
            config={
                "response_mime_type": "application/json",
                "response_json_schema": VibrationalModeList.model_json_schema()
            }
        )
        vibrational_modes = VibrationalModeList.model_validate_json(response.text)
        print(vibrational_modes)

        for j in range(len(tables[i]["rows"])):
            tables[i]["rows"][j]["vibrational_mode_details"] = vibrational_modes.model_dump()["vibrational_modes"][j]
        # Write the updated tables list to the output file
        # Using the same path object as from the previous cell
        with open(path, "w", encoding="utf-8") as f:
            json.dump(existing, f, indent=2)
            f.write("\n")

except Exception as e:
    print(f"Error: {e}")



[{'table_title': 'Table I .', 'table_description': 'The Raman Spectrum of  $\\text{CF}_3\\text{CH}_2\\text{F}$  Liquid (From densitometer tracing of 45 hours exposure)', 'rows': [{'peak': 94, 'peak_high': None, 'peak_low': None, 'peak_width': None, 'relative_intensity': 'w', 'additional_notes': 'Exciting Hg Line: k, Polarization: 0.7'}, {'peak': 289, 'peak_high': None, 'peak_low': None, 'peak_width': None, 'relative_intensity': 's', 'additional_notes': 'Exciting Hg Line: k, Polarization: 0.4'}, {'peak': 142, 'peak_high': None, 'peak_low': None, 'peak_width': None, 'relative_intensity': 'm', 'additional_notes': 'Exciting Hg Line: i, Polarization: 0.6'}, {'peak': 289, 'peak_high': None, 'peak_low': None, 'peak_width': None, 'relative_intensity': 's', 'additional_notes': 'Exciting Hg Line: i, Polarization: 0.4'}, {'peak': 513, 'peak_high': None, 'peak_low': None, 'peak_width': None, 'relative_intensity': 'm', 'additional_notes': 'Exciting Hg Line: k, Polarization: 0.4'}, {'peak': 626, 'pe